# Part 1 — Step 4: Model Evaluation

Compares YOLOv8 and Faster R-CNN on the **held-out test set** using a unified pycocotools COCO evaluation pipeline.

| Metric | Meaning |
|--------|---------|
| mAP@0.5 | Mean Average Precision at IoU=0.5 (primary metric) |
| mAP@0.5:0.95 | COCO standard: averaged over IoU 0.5→0.95 |
| Precision | TP / (TP+FP) at conf=0.25, IoU≥0.5 |
| Recall | TP / (TP+FN) at conf=0.25, IoU≥0.5 |
| Inference (ms) | Wall-clock time per image on the current device |

In [ ]:
# Fix working directory so relative paths work in both Colab and locally
import os
from pathlib import Path

notebook_dir = Path(__file__).parent if '__file__' in dir() else Path.cwd()
repo_root = notebook_dir.parent if (notebook_dir / '../data').exists() else notebook_dir
os.chdir(repo_root)
print(f"Working directory: {os.getcwd()}")

In [ ]:
import json, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline

import torch
from PIL import Image
from torchvision import transforms
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights, FastRCNNPredictor)
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from ultralytics import YOLO
import pandas as pd

In [ ]:
DATA_DIR  = Path("data/acne04")
YOLO_YAML = Path("data/acne04_yolo/dataset.yaml")
YOLO_OUT  = Path("outputs/yolov8")
FRCNN_OUT = Path("outputs/faster_rcnn")
OUT_DIR   = Path("outputs/figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CONF    = 0.25   # shared confidence threshold
device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 1. Evaluate YOLOv8

YOLOv8's built-in `val()` computes all metrics automatically.  
Note: training reported val-set mAP@0.5 = 0.2137; this runs on the held-out **test** set.

In [ ]:
ann_path_test = DATA_DIR / 'test' / '_annotations.coco.json'
coco_gt_test  = COCO(str(ann_path_test))
with open(ann_path_test) as f:
    coco_test_data = json.load(f)

best_yolo_ckpt = YOLO_OUT / 'acne04' / 'weights' / 'best.pt'
best_yolo      = YOLO(str(best_yolo_ckpt))

yolo_metrics = best_yolo.val(data=str(YOLO_YAML), split='test', conf=CONF, verbose=False)
yolo_results = {
    'mAP@50':    float(yolo_metrics.box.map50),
    'mAP@50-95': float(yolo_metrics.box.map),
    'Precision': float(yolo_metrics.box.mp),
    'Recall':    float(yolo_metrics.box.mr),
}

# Inference time (average over 20 images)
_test_imgs = [DATA_DIR / 'test' / m['file_name'] for m in coco_test_data['images'][:20]]
t0 = time.perf_counter()
for _p in _test_imgs:
    best_yolo.predict(str(_p), conf=CONF, verbose=False)
yolo_results['Inference_ms'] = round((time.perf_counter() - t0) / len(_test_imgs) * 1000, 1)

# Collect all predictions at very low conf — used for PR curve & IoU distribution
yolo_all_preds = []
for img_meta in coco_test_data['images']:
    res = best_yolo.predict(str(DATA_DIR / 'test' / img_meta['file_name']),
                            conf=0.001, verbose=False)[0]
    for box, cls, score in zip(res.boxes.xyxy.tolist(),
                                res.boxes.cls.tolist(),
                                res.boxes.conf.tolist()):
        x1, y1, x2, y2 = box
        yolo_all_preds.append({'image_id': img_meta['id'],
                               'bbox': [x1, y1, x2-x1, y2-y1],
                               'score': float(score),
                               'category_id': coco_test_data['categories'][int(cls)]['id']})

print(f"YOLOv8s  mAP@0.5={yolo_results['mAP@50']:.4f}  "
      f"P={yolo_results['Precision']:.4f}  R={yolo_results['Recall']:.4f}  "
      f"inf={yolo_results['Inference_ms']}ms")

## 2. Evaluate Faster R-CNN

Run inference on the test set, then score with `pycocotools COCOeval`.  
Precision and Recall are computed by matching predictions to GT at IoU≥0.5  
(NOT `evaluator.stats[6]` which is AR@1, not threshold-fixed recall).

In [ ]:
frcnn_eval = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
frcnn_eval.roi_heads.box_predictor = FastRCNNPredictor(
    frcnn_eval.roi_heads.box_predictor.cls_score.in_features, 5)
frcnn_eval.load_state_dict(torch.load(str(FRCNN_OUT / 'best.pth'),
                                      map_location=device, weights_only=False))
frcnn_eval.to(device).eval()

cats = sorted(coco_test_data['categories'], key=lambda c: c['id'])
label_to_cat = {i+1: c['id'] for i, c in enumerate(cats)}
tf_img = transforms.ToTensor()

# Collect all predictions at very low conf
frcnn_all_preds = []
for img_id, img_meta in coco_gt_test.imgs.items():
    img    = Image.open(DATA_DIR / 'test' / img_meta['file_name']).convert('RGB')
    tensor = tf_img(img).unsqueeze(0).to(device)
    with torch.no_grad():
        out = frcnn_eval(tensor)[0]
    for box, lbl, score in zip(out['boxes'], out['labels'], out['scores']):
        if score.item() < 0.001: continue
        x1, y1, x2, y2 = box.tolist()
        frcnn_all_preds.append({'image_id': img_id,
                                'category_id': label_to_cat.get(lbl.item(), lbl.item()),
                                'bbox': [x1, y1, x2-x1, y2-y1],
                                'score': score.item()})

# Inference time
t0 = time.perf_counter()
for img_id, img_meta in list(coco_gt_test.imgs.items())[:20]:
    img = Image.open(DATA_DIR / 'test' / img_meta['file_name']).convert('RGB')
    with torch.no_grad():
        frcnn_eval(tf_img(img).unsqueeze(0).to(device))
frcnn_ms = round((time.perf_counter() - t0) / 20 * 1000, 1)

# pycocotools mAP
coco_dt   = coco_gt_test.loadRes(frcnn_all_preds) if frcnn_all_preds else coco_gt_test.loadRes([])
evaluator = COCOeval(coco_gt_test, coco_dt, 'bbox')
evaluator.evaluate(); evaluator.accumulate(); evaluator.summarize()

# Correct P/R via greedy IoU matching (not stats[6]=AR@1)
def _box_iou(a, b):
    ax1,ay1,aw,ah = a; ax2,ay2 = ax1+aw, ay1+ah
    bx1,by1,bw,bh = b; bx2,by2 = bx1+bw, by1+bh
    ix = max(0, min(ax2,bx2) - max(ax1,bx1))
    iy = max(0, min(ay2,by2) - max(ay1,by1))
    inter = ix * iy
    union = aw*ah + bw*bh - inter
    return inter/union if union > 0 else 0.0

gt_by_img = {img_id: coco_gt_test.loadAnns(coco_gt_test.getAnnIds(imgIds=img_id))
             for img_id in coco_gt_test.imgs}
total_gt  = sum(len(v) for v in gt_by_img.values())
matched   = {img_id: [False]*len(anns) for img_id, anns in gt_by_img.items()}

tp = fp = 0
for p in sorted([x for x in frcnn_all_preds if x['score'] >= CONF], key=lambda x: -x['score']):
    img_id = p['image_id']
    best_iou, best_j = 0.0, -1
    for j, gt_ann in enumerate(gt_by_img.get(img_id, [])):
        if matched[img_id][j]: continue
        iou = _box_iou(p['bbox'], gt_ann['bbox'])
        if iou > best_iou: best_iou, best_j = iou, j
    if best_iou >= 0.5 and best_j >= 0:
        matched[img_id][best_j] = True; tp += 1
    else:
        fp += 1
fn = total_gt - tp

frcnn_results = {
    'mAP@50':       float(evaluator.stats[1]),
    'mAP@50-95':    float(evaluator.stats[0]),
    'Precision':    tp / (tp + fp) if tp + fp > 0 else 0.0,
    'Recall':       tp / (tp + fn) if tp + fn > 0 else 0.0,
    'Inference_ms': frcnn_ms,
}
print(f"Faster R-CNN  mAP@0.5={frcnn_results['mAP@50']:.4f}  "
      f"P={frcnn_results['Precision']:.4f}  R={frcnn_results['Recall']:.4f}  "
      f"inf={frcnn_results['Inference_ms']}ms")

## 3. Comparison Table

In [ ]:
rows = []
for name, res in [('YOLOv8s', yolo_results), ('Faster R-CNN', frcnn_results)]:
    rows.append({'Model': name,
                 'mAP@50':    res['mAP@50'],
                 'mAP@50-95': res['mAP@50-95'],
                 'Precision': res['Precision'],
                 'Recall':    res['Recall'],
                 'Inf. (ms)': res['Inference_ms']})

df = pd.DataFrame(rows).set_index('Model')
display(df.style.format("{:.4f}", na_rep="—").highlight_max(axis=0, color="#d4edda"))

OUT_FILE = Path("outputs/evaluation_results.json")
OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
with open(OUT_FILE, "w") as f:
    json.dump({"yolov8": yolo_results, "faster_rcnn": frcnn_results}, f, indent=2)
print(f"Saved → {OUT_FILE}")

## 4. Precision-Recall Curves

PR curves computed over all test-set detections sorted by confidence, matched to GT at IoU=0.5.  
Area under each curve ≈ mAP@0.5.

In [ ]:
def pr_curve(all_preds, coco_gt_obj, iou_thresh=0.5):
    gt_by_img = {img_id: coco_gt_obj.loadAnns(coco_gt_obj.getAnnIds(imgIds=img_id))
                 for img_id in coco_gt_obj.imgs}
    total_gt = sum(len(v) for v in gt_by_img.values())
    matched  = {img_id: [False]*len(anns) for img_id, anns in gt_by_img.items()}
    tp_list  = []
    for p in sorted(all_preds, key=lambda x: -x['score']):
        img_id = p['image_id']
        best_iou, best_j = 0.0, -1
        for j, gt_ann in enumerate(gt_by_img.get(img_id, [])):
            if matched[img_id][j]: continue
            iou = _box_iou(p['bbox'], gt_ann['bbox'])
            if iou > best_iou: best_iou, best_j = iou, j
        if best_iou >= iou_thresh and best_j >= 0:
            matched[img_id][best_j] = True; tp_list.append(1)
        else:
            tp_list.append(0)
    tp_c = np.cumsum(tp_list)
    fp_c = np.cumsum([1 - x for x in tp_list])
    return tp_c / (tp_c + fp_c + 1e-9), tp_c / (total_gt + 1e-9)

yolo_p,  yolo_r  = pr_curve(yolo_all_preds,  coco_gt_test)
frcnn_p, frcnn_r = pr_curve(frcnn_all_preds, coco_gt_test)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(yolo_r,  yolo_p,  label=f'YOLOv8s (mAP={yolo_results["mAP@50"]:.3f})',  color='#2196F3', lw=2)
ax.plot(frcnn_r, frcnn_p, label=f'Faster R-CNN (mAP={frcnn_results["mAP@50"]:.3f})', color='#FF5722', lw=2)
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title('Precision-Recall Curves — ACNE04 Test Set (IoU=0.5)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'pr_curves.png', dpi=150)
plt.show()

## 5. IoU Distribution

Per-detection max IoU with the nearest GT box, at confidence ≥ 0.25.

In [ ]:
def iou_distribution(all_preds, coco_gt_obj, conf_thresh=CONF):
    gt_by_img = {img_id: coco_gt_obj.loadAnns(coco_gt_obj.getAnnIds(imgIds=img_id))
                 for img_id in coco_gt_obj.imgs}
    ious = []
    for p in all_preds:
        if p['score'] < conf_thresh: continue
        best_iou = max((_box_iou(p['bbox'], gt['bbox'])
                        for gt in gt_by_img.get(p['image_id'], [])), default=0.0)
        ious.append(best_iou)
    return np.array(ious)

yolo_ious  = iou_distribution(yolo_all_preds,  coco_gt_test)
frcnn_ious = iou_distribution(frcnn_all_preds, coco_gt_test)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, ious, name in zip(axes, [yolo_ious, frcnn_ious], ['YOLOv8s', 'Faster R-CNN']):
    pct = 100 * np.mean(ious >= 0.5)
    ax.hist(ious, bins=25, range=(0, 1), color='#4CAF50', edgecolor='white', alpha=0.85)
    ax.axvline(0.5, color='red', linestyle='--', lw=1.5, label='IoU=0.5 threshold')
    ax.set_xlabel('Max IoU with GT box'); ax.set_ylabel('Detections')
    ax.set_title(f'{name}\n{len(ious)} dets (conf≥{CONF})  mean={np.mean(ious):.3f}  {pct:.1f}% matched')
    ax.legend(fontsize=8)

plt.suptitle('IoU Distribution — ACNE04 Test Set', fontsize=12)
plt.tight_layout()
plt.savefig(OUT_DIR / 'iou_distribution.png', dpi=150)
plt.show()
print(f'YOLOv8 : mean IoU={np.mean(yolo_ious):.3f},  {100*np.mean(yolo_ious>=0.5):.1f}% above 0.5')
print(f'F-RCNN : mean IoU={np.mean(frcnn_ious):.3f}, {100*np.mean(frcnn_ious>=0.5):.1f}% above 0.5')

## 6. Per-Lesion-Density mAP

mAP@0.5 broken down by annotation density bin.  
Per-bin mAP is computed by running pycocotools on each subset independently;  
values can exceed overall mAP because the per-subset PR curve excludes cross-subset false positives.

In [ ]:
import contextlib, io

ann_count_by_img = {}
for ann in coco_test_data['annotations']:
    ann_count_by_img[ann['image_id']] = ann_count_by_img.get(ann['image_id'], 0) + 1

def bin_label(n):
    if n <= 5:  return 'Low (1-5)'
    if n <= 15: return 'Med (6-15)'
    return 'High (16+)'

bins = {'Low (1-5)': [], 'Med (6-15)': [], 'High (16+)': []}
for img_id, n in ann_count_by_img.items():
    bins[bin_label(n)].append(img_id)

print('Density bins (test set):')
for k, v in bins.items():
    print(f'  {k}: {len(v)} images')

def map50_for_ids(dt_obj, gt_obj, img_ids):
    if not img_ids: return float('nan')
    ev = COCOeval(gt_obj, dt_obj, 'bbox')
    ev.params.imgIds = img_ids
    ev.evaluate(); ev.accumulate()
    with contextlib.redirect_stdout(io.StringIO()):
        ev.summarize()  # required — populates ev.stats
    return float(ev.stats[1])

yolo_coco_dt  = coco_gt_test.loadRes(yolo_all_preds)
frcnn_coco_dt = coco_gt_test.loadRes(frcnn_all_preds)

rows_density = []
for bin_name, img_ids in bins.items():
    rows_density.append({
        'Density': bin_name,
        'Images':  len(img_ids),
        'YOLOv8s mAP@50':     round(map50_for_ids(yolo_coco_dt,  coco_gt_test, img_ids), 4),
        'Faster R-CNN mAP@50': round(map50_for_ids(frcnn_coco_dt, coco_gt_test, img_ids), 4),
    })

df_density = pd.DataFrame(rows_density).set_index('Density')
display(df_density.style.format("{:.4f}").highlight_max(axis=0,
        subset=['YOLOv8s mAP@50', 'Faster R-CNN mAP@50'], color='#d4edda'))

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(bins)); w = 0.35
yolo_vals  = [r['YOLOv8s mAP@50']     for r in rows_density]
frcnn_vals = [r['Faster R-CNN mAP@50'] for r in rows_density]
ax.bar(x - w/2, yolo_vals,  w, label='YOLOv8s',      color='#2196F3', alpha=0.85)
ax.bar(x + w/2, frcnn_vals, w, label='Faster R-CNN', color='#FF5722', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(list(bins.keys()))
ax.set_ylabel('mAP@0.5')
ax.set_ylim(0, max(max(yolo_vals + frcnn_vals, default=0) * 1.4, 0.05))
ax.set_title('mAP@0.5 by Lesion Density — ACNE04 Test Set')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'density_map.png', dpi=150)
plt.show()